# Proceso ETL - Datos de Conectividad en Argentina

Este notebook implementa el proceso de Extracción, Transformación y Carga (ETL) para los datos de conectividad en Argentina.

In [6]:
import pandas as pd
import numpy as np
import glob
import os
import logging
from datetime import datetime
from pathlib import Path

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_process.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuración de directorios
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = DATA_DIR

# Crear directorios si no existen
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

In [7]:
def clean_column_names(df):
    """Limpia los nombres de las columnas del DataFrame."""
    df.columns = df.columns.str.lower()
    df.columns = df.columns.str.replace(' ', '_')
    df.columns = df.columns.str.replace('[^a-z0-9_]', '')
    return df

def standardize_date(df, date_columns):
    """Estandariza el formato de fecha en las columnas especificadas."""
    for col in date_columns:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col])
            except Exception as e:
                logger.error(f"Error al convertir la columna {col} a fecha: {str(e)}")
    return df

def clean_numeric_columns(df):
    """Limpia las columnas numéricas del DataFrame."""
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def handle_missing_values(df):
    """Maneja los valores faltantes en el DataFrame."""
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    categorical_columns = df.select_dtypes(include=['object']).columns
    
    # Para columnas numéricas, usar la mediana
    df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())
    
    # Para columnas categóricas, usar el modo
    for col in categorical_columns:
        df[col] = df[col].fillna(df[col].mode().iloc[0] if not df[col].mode().empty else 'DESCONOCIDO')
    
    return df

In [8]:
def load_csv_files(folder_path):
    """Carga todos los archivos CSV del directorio especificado."""
    datasets = {}
    
    try:
        csv_files = list(Path(folder_path).glob('*.csv'))
        logger.info(f"Encontrados {len(csv_files)} archivos CSV")
        
        for file_path in csv_files:
            try:
                logger.info(f"Procesando archivo: {file_path.name}")
                
                # Leer el archivo CSV
                df = pd.read_csv(file_path)
                
                # Aplicar transformaciones
                df = clean_column_names(df)
                df = standardize_date(df, ['fecha', 'periodo'])
                df = clean_numeric_columns(df)
                df = handle_missing_values(df)
                
                # Guardar el DataFrame procesado
                datasets[file_path.stem] = df
                logger.info(f"Archivo {file_path.name} procesado exitosamente")
                
            except Exception as e:
                logger.error(f"Error al procesar el archivo {file_path.name}: {str(e)}")
                continue
                
    except Exception as e:
        logger.error(f"Error al acceder al directorio {folder_path}: {str(e)}")
        
    return datasets

def save_to_parquet(datasets, output_folder):
    """Guarda los DataFrames en formato Parquet."""
    for name, df in datasets.items():
        try:
            output_path = Path(output_folder) / f"{name}.parquet"
            df.to_parquet(output_path, index=False)
            logger.info(f"Archivo {output_path.name} guardado exitosamente")
        except Exception as e:
            logger.error(f"Error al guardar el archivo {name}.parquet: {str(e)}")

In [9]:
# Ejecutar el proceso ETL
logger.info("Iniciando proceso ETL...")

# Cargar datos
datasets = load_csv_files(DATA_DIR)

# Guardar datos procesados
save_to_parquet(datasets, OUTPUT_DIR)

logger.info("Proceso ETL completado exitosamente")

2025-03-30 19:32:51,861 - INFO - Iniciando proceso ETL...
2025-03-30 19:32:51,862 - INFO - Encontrados 9 archivos CSV
2025-03-30 19:32:51,864 - INFO - Procesando archivo: Internet Accesos Tecnologia Localidades.csv


2025-03-30 19:32:51,921 - INFO - Archivo Internet Accesos Tecnologia Localidades.csv procesado exitosamente
2025-03-30 19:32:51,922 - INFO - Procesando archivo: Internet Accesos Tecnologia Provincias.csv
2025-03-30 19:32:51,939 - INFO - Archivo Internet Accesos Tecnologia Provincias.csv procesado exitosamente
2025-03-30 19:32:51,940 - INFO - Procesando archivo: Internet Accesos Tecnologia Totales.csv
2025-03-30 19:32:51,957 - INFO - Archivo Internet Accesos Tecnologia Totales.csv procesado exitosamente
2025-03-30 19:32:51,959 - INFO - Procesando archivo: Internet Accesos Velocidad Rango Provincias.csv
2025-03-30 19:32:51,977 - INFO - Archivo Internet Accesos Velocidad Rango Provincias.csv procesado exitosamente
2025-03-30 19:32:51,978 - INFO - Procesando archivo: Internet BAF Provincias.csv
2025-03-30 19:32:51,995 - INFO - Archivo Internet BAF Provincias.csv procesado exitosamente
2025-03-30 19:32:51,997 - INFO - Procesando archivo: Internet Ingresos.csv
2025-03-30 19:32:52,008 - INFO 

## Resumen de Datos Procesados

In [10]:
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print(f"Forma: {df.shape}")
    print("Columnas:")
    for col in df.columns:
        print(f"  - {col}: {df[col].dtype}")


Dataset: Internet Accesos Tecnologia Localidades
Forma: (10509, 6)
Columnas:
  - linkindec: int64
  - provincia: object
  - partido: object
  - localidad: object
  - tecnologia: object
  - accesos: int64

Dataset: Internet Accesos Tecnologia Provincias
Forma: (1032, 9)
Columnas:
  - año: int64
  - trimestre: int64
  - provincia: object
  - adsl: int64
  - cablemodem: int64
  - fibra_optica: int64
  - wireless: int64
  - otros: int64
  - total: int64

Dataset: Internet Accesos Tecnologia Totales
Forma: (43, 8)
Columnas:
  - año: int64
  - trimestre: int64
  - adsl: int64
  - cablemodem: int64
  - fibra_optica: int64
  - wireless: int64
  - otros: int64
  - total: int64

Dataset: Internet Accesos Velocidad Rango Provincias
Forma: (1024, 12)
Columnas:
  - año: int64
  - trimestre: int64
  - provincia: object
  - hasta_512_kbps: int64
  - +_512_kbps_-_1_mbps: int64
  - +_1_mbps_-_6_mbps: int64
  - +_6_mbps_-_10_mbps: int64
  - +_10_mbps_-_20_mbps: int64
  - +_20_mbps_-_30_mbps: int64
  - 